In [ ]:
#경고 표시 줄이기
import warnings
warnings.filterwarnings('ignore')

# package install
!pip install -q pip
!pip install -q -U langchain_google_genai
!pip install -q -U google-ai-generativelanguage
!pip install -q openpyxl

In [27]:
# 데이터셋 불러오기
import pandas as pd
import openpyxl

df = pd.read_excel("revision_240812_dataset.xlsx")

In [ ]:
# gpt4o-mini 설정
import openai
from openai import OpenAI

# OPENAI_API_KEY = 

client = OpenAI(api_key=OPENAI_API_KEY)

model = "gpt-4o-mini"

system_verify_prompt = "당신은 문장 비교 전문가입니다. 두 문장을 읽고 같은 의미인지 다른 의미인지 판별해 주십시오. 같으면 T 다르면 F를 출력해 주십시오."

In [ ]:
# Gemini 설정

import os
from langchain_google_genai import ChatGoogleGenerativeAI

# os.environ["GOOGLE_API_KEY"] = 

llm = ChatGoogleGenerativeAI(
    model="gemini-pro",
    convert_system_message_to_human=True
)

In [30]:
# 입력 : 비교할 두 문장 / 출력 : T or F

def verification(sentence1, sentence2):
    # 메시지 설정하기
    messages = [{
        "role": "system",
        "content": system_verify_prompt
    }, {
        "role": "user",
        "content": sentence1 + sentence2
    }]
    
    response = client.chat.completions.create(model=model, messages=messages)
    answer = response.choices[0].message.content
    print(answer, end = "\t")
    return answer

In [31]:
import json

def extract_augsentence(json_string):
    try:
        # JSON 문자열을 파이썬 딕셔너리로 변환
        data = json.loads(json_string)
        
        # '교체 문장' 키의 값을 반환
        return data.get("출력", None)
    except json.JSONDecodeError:
        # JSON 파싱 에러 처리
        print("유효하지 않은 JSON 형식입니다.")
        return None

In [36]:
import time

prompt = """
데이터 증강을 하려고 해. 주어진 문장에서 최대 두 개의 단어를 동의어로 교체해 줘. 
출력 형식은 아래와 같은 JSON 형식이야. 

{"출력": "{교체된 문장}"}
"""

# 입력 : 프롬프트&원본 문장 / 출력 : 교체 문장 or None
def sentence_aug(sentence, max_retries=3):
    retries = 0
    while retries < max_retries:
        # LLM 호출 및 교체 문장 추출
        text = llm.invoke(prompt + sentence)
        text = text.content
        aug_sentence = extract_augsentence(text)

        # 유사도 검증
        similarity = verification(sentence, aug_sentence)
        if 'T' in similarity:
            print(f"aug_sentence : {aug_sentence}")
            return aug_sentence
        else:
            retries += 1
            print(f"aug_sentence : {aug_sentence}", end = "")S
            print(f"\tAttempt {retries}/{max_retries}")
    time.sleep(200)
    
    # 최대 시도 횟수에 도달한 경우
    print("Failed to generate")
    return None

In [37]:
# 1. input_sentence augmentation

# 입력 : 원본 df / 출력 : 증강 df
max_augments = 2

for idx, row in df.iterrows():
    print("\n\n=====")
    print(f"input_sentence : {row['input_sentence']}")
    
    aug_count = 0
    try_count = 0
    while aug_count < max_augments:
        aug_sentence = sentence_aug(row['input_sentence'])
        if aug_sentence:
            augmented_row = row.copy()
            augmented_row['input_sentence'] = aug_sentence
            df = pd.concat([df, pd.DataFrame([augmented_row])], ignore_index=True)
            aug_count += 1
        else:
            print(f"Failed to generate augmented sentence on attempt {aug_count + 1}.")
            break



=====
input_sentence : 신규 고객 유치 (매출 300% 달성 목표)
F	aug_sentence : 새로운 고객 확보 (수익 300% 증가 목표)	Attempt 1/3
T	aug_sentence : 새로운 고객 획득 (매출 300% 증대 목표)
F	aug_sentence : 신규 고객 확보 (매출 300% 달성 목표)	Attempt 1/3
T	aug_sentence : 신규 고객 확보 (매출 300% 달성 목표)


=====
input_sentence : APP 다운로드 5만 달성 (500% 성장)


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..


T	aug_sentence : APP 다운로드 5만 달성 (5배 성장)


ResourceExhausted: 429 Resource has been exhausted (e.g. check quota).

In [ ]:
df.to_excel('240814_dataset_input_augmented.xlsx', index=True, engine='openpyxl')